## Imports

In [10]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from datasets import load_dataset
from transformers import BertTokenizer, BertModel, DataCollatorWithPadding
from torch.optim import AdamW

# Load dataset

In [11]:
dataset = load_dataset("stanfordnlp/imdb")

dataset["train"] = dataset["train"].shuffle(seed=42).select(range(10000))
dataset["test"] = dataset["test"].shuffle(seed=42).select(range(1000))


## Load tokenizer and model

In [12]:
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

def tokenize(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=256
    )

dataset = dataset.map(tokenize, batched=True)

dataset = dataset.rename_column("label", "labels")
dataset.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

## DataLoader

In [13]:
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

train_loader = DataLoader(
    dataset["train"],
    batch_size=16,
    shuffle=True,
    collate_fn=data_collator
)

test_loader = DataLoader(
    dataset["test"],
    batch_size=16,
    shuffle=False,
    collate_fn=data_collator
)

# Model: Frozen BERT + MLP

In [ ]:
class BertMLP(nn.Module):
    def __init__(self):
        super().__init__()

        self.bert = BertModel.from_pretrained("bert-base-uncased")

        # freeze BERT
        for param in self.bert.parameters():
            param.requires_grad = False

        self.classifier = nn.Sequential(
            nn.Linear(768, 256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, 64),
            nn.ReLU(),
            nn.Linear(64, 2)
        )

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        cls_token = outputs.last_hidden_state[:, 0]  # CLS token
        return self.classifier(cls_token)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = BertMLP().to(device)

criterion = nn.CrossEntropyLoss()
optimizer = AdamW(model.classifier.parameters(), lr=2e-4)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 14705.71it/s]
[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


# Training

In [ ]:
def train_one_epoch():
    model.train()
    total_loss = 0

    for batch in train_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        optimizer.zero_grad()

        logits = model(input_ids, attention_mask)
        loss = criterion(logits, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(train_loader)

# Evaluation

In [ ]:

def evaluate():
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for batch in test_loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            logits = model(input_ids, attention_mask)
            preds = torch.argmax(logits, dim=1)

            correct += (preds == labels).sum().item()
            total += labels.size(0)

    return correct / total

# Train loop

In [ ]:
for epoch in range(3):
    loss = train_one_epoch()
    acc = evaluate()

    print(f"Epoch {epoch+1}")
    print(f"Loss: {loss:.4f}")
    print(f"Accuracy: {acc:.4f}")
    print("----------------------")

Epoch 1
Loss: 0.4820
Accuracy: 0.8060
----------------------
Epoch 2
Loss: 0.4127
Accuracy: 0.8070
----------------------
Epoch 3
Loss: 0.4048
Accuracy: 0.8290
----------------------
